# 2) base_fft_FFT — Qwen3-1.7B-Base Full Fine-Tune через FSDP

## Changelog
- **v36** — откат fp16-save: сохраняем fp32 (master веса). Rotation-before-save одна решает disk-full: пик `keep_last × 8 GB = 16 GB` < 20 GB лимит. `save_final_hf` чистит ckpt перед final.
- **v35** — disk fix через fp16 (откачено: не нужно).
- **v34** — `Qwen/Qwen3-1.7B-Base` (настоящая pretrained-only).
- **v31** — `empty_cache` после save (OOM fix).
- **v30** — skip optim_state (bnb+FSDP крашатся).
- **v29** — убран `UserSecretsClient.get_secret`.
- **v28** — auto-detect dataset через `rglob`.
- **v27** — dataset attached через CLI push.
- **v26** — диагностическая ячейка.
- **v25** — rotation чекпоинтов.

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
print(f"=== rglob '{INPUT_DIR}' (все файлы) ===")
for p in sorted(INPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size / 1e6:.2f} MB)")

train_hits = list(INPUT_DIR.rglob("train.jsonl"))
val_hits = list(INPUT_DIR.rglob("val.jsonl"))
print(f"\ntrain.jsonl: {train_hits}")
print(f"val.jsonl:   {val_hits}")

assert train_hits, "train.jsonl не найден в /kaggle/input/** — dataset не подключён"
assert val_hits, "val.jsonl не найден в /kaggle/input/**"

TRAIN_JSONL = train_hits[0]
VAL_JSONL = val_hits[0]
print(f"\nwill use:\n  TRAIN: {TRAIN_JSONL}\n  VAL:   {VAL_JSONL}")


## 1. Setup

In [ ]:
!pip install bitsandbytes -q 

In [ ]:
from pathlib import Path

Path("/kaggle/working/src").mkdir(parents=True, exist_ok=True)

!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} — {mem:.1f} GB")

## 2. Запись модулей в `/kaggle/working/src/` через `%%writefile`

Каждый модуль — self-contained: свои импорты (дублируются между файлами) + функции.
`train_sft.py` импортирует из соседних модулей через `from utils import ...` — это работает, потому что `torchrun` кладёт директорию скрипта в `sys.path[0]`.

### `src/utils.py` — Logger + set_seed

In [ ]:
%%writefile /kaggle/working/src/utils.py
"""Переиспользуемые утилиты: Logger + set_seed."""
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


def set_seed(seed: int) -> None:
    """Детерминизм. Вызывать ПОСЛЕ dist.init_process_group."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class Logger:
    """tqdm-progress bar с ETA + сбор train/eval метрик + финальный JSON.
    Только rank 0 реально что-то делает; остальные ранки no-op."""

    def __init__(self, rank: int, total_steps: int, out_path=None):
        self._rank = rank
        self._total = total_steps
        self._entries = []
        self._out_path = out_path
        self._t_last = time.time()
        self._pbar = None
        if rank == 0:
            self._pbar = tqdm(
                total=total_steps,
                desc="train",
                unit="step",
                dynamic_ncols=True,
                mininterval=0.5,
            )

    def log_train(self, step, loss, lr, gnorm):
        if self._rank != 0:
            return
        now = time.time()
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        entry = {
            "type": "train",
            "step": step,
            "loss": float(loss),
            "lr": float(lr),
            "gnorm": float(gnorm),
            "peak_mem_gb": round(peak_mem, 2),
            "dt_s": round(now - self._t_last, 2),
        }
        self._entries.append(entry)
        self._pbar.n = step
        self._pbar.set_postfix(
            loss=f"{entry['loss']:.4f}",
            lr=f"{entry['lr']:.2e}",
            gnorm=f"{entry['gnorm']:.3f}",
            mem=f"{peak_mem:.1f}G",
        )
        self._pbar.refresh()
        self._t_last = now
        torch.cuda.reset_peak_memory_stats()

    def log_eval(self, step, val_loss):
        if self._rank != 0:
            return
        entry = {"type": "eval", "step": step, "val_loss": float(val_loss)}
        self._entries.append(entry)
        tqdm.write(f"[step {step:>5}] val_loss={entry['val_loss']:.4f}")

    def save(self):
        if self._rank != 0 or self._out_path is None:
            return
        self._out_path.write_text(
            json.dumps(self._entries, indent=2, ensure_ascii=False)
        )
        tqdm.write(f"[rank0] metrics saved → {self._out_path}")

    def close(self):
        if self._pbar is not None:
            self._pbar.close()

    @property
    def entries(self):
        return list(self._entries)


### `src/data.py` — Dataset + Collator + build_dataloaders

In [ ]:
%%writefile /kaggle/working/src/data.py
"""Data pipeline: JSONL-dataset с ChatML/labels-masking, collator, dataloaders."""
from __future__ import annotations

import json

import torch
from torch.utils.data import DataLoader, Dataset, DistributedSampler


class ChatJsonlDataset(Dataset):

    def __init__(self, filepath, tokenizer, max_seq_length):
        self.tokenized_dataset = []
        self.source_dataset = []

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                self.source_dataset.append(json.loads(line))

        self._tokenizer = tokenizer
        self._max_seq_length = max_seq_length

        self._build_dataset()
        self.source_dataset = None

    def _build_dataset(self):
        for ex in self.source_dataset:
            messages = ex["messages"]
            if not messages or messages[-1]["role"] != "assistant":
                continue

            full_text = self._tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
            prefix_text = self._tokenizer.apply_chat_template(
                messages[:-1], tokenize=False, add_generation_prompt=True,
            )

            full_ids = self._tokenizer.encode(
                full_text,
                add_special_tokens=False,
                truncation=True,
                max_length=self._max_seq_length,
            )
            prefix_len = len(
                self._tokenizer.encode(prefix_text, add_special_tokens=False)
            )

            if prefix_len >= len(full_ids):
                continue

            labels = list(full_ids)
            for i in range(prefix_len):
                labels[i] = -100

            self.tokenized_dataset.append({
                "input_ids": full_ids,
                "labels": labels,
                "attention_mask": [1] * len(full_ids),
            })

    def __len__(self):
        return len(self.tokenized_dataset)

    def __getitem__(self, idx):
        return self.tokenized_dataset[idx]


class PadCollator:

    def __init__(self, pad_token_id):
        self._pad_token_id = pad_token_id

    def __call__(self, batch):
        batch = [b for b in batch if b is not None]
        if not batch:
            return None

        max_len = max(len(b["input_ids"]) for b in batch)

        def pad(seq, pad_val):
            return seq + [pad_val] * (max_len - len(seq))

        return {
            "input_ids":      torch.tensor([pad(b["input_ids"],      self._pad_token_id) for b in batch], dtype=torch.long),
            "labels":         torch.tensor([pad(b["labels"],         -100)               for b in batch], dtype=torch.long),
            "attention_mask": torch.tensor([pad(b["attention_mask"], 0)                  for b in batch], dtype=torch.long),
        }


def build_dataloaders(train_path, val_path, tokenizer,
                      max_seq_length, per_device_bs, seed, rank, world_size):
    collator = PadCollator(tokenizer.pad_token_id)

    train_ds = ChatJsonlDataset(train_path, tokenizer, max_seq_length)
    val_ds = ChatJsonlDataset(val_path, tokenizer, max_seq_length)

    train_sampler = DistributedSampler(
        train_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=seed,
    )
    val_sampler = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank, shuffle=False,
    )

    train_loader = DataLoader(
        train_ds, batch_size=per_device_bs, sampler=train_sampler,
        collate_fn=collator, num_workers=0, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=per_device_bs, sampler=val_sampler,
        collate_fn=collator, num_workers=0, pin_memory=True,
    )
    return train_loader, val_loader, train_sampler


### `src/model.py` — load_model + FSDP wrap + optimizer + evaluate

In [ ]:
%%writefile /kaggle/working/src/model.py
"""Model, FSDP wrap, optimizer+scheduler, eval-loop."""
from __future__ import annotations

import functools

import bitsandbytes as bnb
import torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    BackwardPrefetch,
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3.modeling_qwen3 import Qwen3DecoderLayer


def load_model_and_tokenizer(model_name, chat_template):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="right",
        chat_template=chat_template,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float32,
        attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    for p in model.parameters():
        p.requires_grad = True
    return tokenizer, model


def wrap_with_fsdp(model, rank):
    mp = MixedPrecision(
        param_dtype=torch.float16,
        reduce_dtype=torch.float32,
        buffer_dtype=torch.float32,
    )
    wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={Qwen3DecoderLayer},
    )
    return FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        auto_wrap_policy=wrap_policy,
        mixed_precision=mp,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
        device_id=rank,
        use_orig_params=True,
        limit_all_gathers=True,
    )


def build_optimizer_and_scheduler(model, lr, weight_decay, warmup_ratio, total_steps):
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    optimizer = bnb.optim.PagedAdamW8bit(
        model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.95),
    )
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])
    return optimizer, scheduler


@torch.no_grad()
def evaluate(model, val_loader, rank, max_batches):
    total_loss = torch.zeros(1, device=rank)
    total_tokens = torch.zeros(1, device=rank)
    model.eval()
    for i, batch in enumerate(val_loader):
        if i >= max_batches:
            break
        if batch is None:
            continue
        batch = {k: v.to(rank) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(**batch)
        n = (batch["labels"] != -100).sum()
        total_loss += n * output.loss
        total_tokens += n
    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
    dist.all_reduce(total_tokens, op=dist.ReduceOp.SUM)
    model.train()
    return (total_loss / total_tokens.clamp(min=1)).item()


### `src/checkpoint.py` — save/load/save_final_hf (с rotation)

In [ ]:
%%writefile /kaggle/working/src/checkpoint.py
"""FSDP-aware save/load + HF-export для eval-ноутбука.

Optim state не сохраняем: bnb.PagedAdamW8bit + FSDP FULL_STATE_DICT падают
с `tensors on cuda:0 and cuda:1` в `_convert_all_state_info`. Resume теряет Adam
moments, но пересчитываются за несколько шагов.

Disk budget на Kaggle: `/kaggle/working/` = 20 GB. Rotate ДО save (не после),
иначе пик во время записи = (keep_last + 1) файлов. С fp32-весами и keep_last=2
пик = 2 × 8 GB = 16 GB, запас ~4 GB.
"""
from __future__ import annotations

import random

import numpy as np
import torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullStateDictConfig,
    FullyShardedDataParallel as FSDP,
    StateDictType,
)


def save_checkpoint(model, optimizer, scheduler, global_step, epoch, rank, ckpt_dir, keep_last=2):
    save_cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)

    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_cfg):
        model_sd = model.state_dict()

    if rank == 0:
        ckpt_dir.mkdir(parents=True, exist_ok=True)

        # Rotate ПЕРЕД записью: оставляем max(keep_last-1) старых,
        # новый save доведёт до keep_last. Пик disk = keep_last × size.
        existing = sorted(
            ckpt_dir.glob("ckpt-step-*.pt"),
            key=lambda p: int(p.stem.rsplit("-", 1)[-1]),
        )
        keep_before = max(0, keep_last - 1)
        to_delete = existing if keep_before == 0 else existing[:-keep_before]
        for p in to_delete:
            p.unlink()

        payload = {
            "model": model_sd,
            "sched": scheduler.state_dict(),
            "step": global_step,
            "epoch": epoch,
            "rng": {
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all(),
                "numpy": np.random.get_state(),
                "py": random.getstate(),
            },
        }
        torch.save(payload, ckpt_dir / f"ckpt-step-{global_step}.pt")

    torch.cuda.empty_cache()
    dist.barrier()


def load_checkpoint(model, optimizer, scheduler, ckpt_path, rank):
    payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    load_cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=False)

    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, load_cfg):
        model.load_state_dict(payload["model"])

    scheduler.load_state_dict(payload["sched"])

    torch.set_rng_state(payload["rng"]["torch"])
    torch.cuda.set_rng_state_all(payload["rng"]["cuda"])
    np.random.set_state(payload["rng"]["numpy"])
    random.setstate(payload["rng"]["py"])

    return payload["step"], payload["epoch"]


def save_final_hf(model, tokenizer, out_dir, rank, ckpt_dir=None):
    save_cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_cfg):
        sd = model.state_dict()

    if rank == 0:
        # Освобождаем место: все ckpt больше не нужны, финальная модель важнее.
        if ckpt_dir is not None and ckpt_dir.exists():
            for p in ckpt_dir.glob("ckpt-step-*.pt"):
                p.unlink()

        out_dir.mkdir(parents=True, exist_ok=True)
        model.module.config.save_pretrained(out_dir)
        sd_fp16 = {k: v.to(torch.float16).contiguous() for k, v in sd.items()}
        torch.save(sd_fp16, out_dir / "pytorch_model.bin")
        tokenizer.save_pretrained(out_dir)

    torch.cuda.empty_cache()
    dist.barrier()


### `src/train_sft.py` — CONFIG + train loop + main (entry point)

In [ ]:
%%writefile /kaggle/working/src/train_sft.py
"""Qwen3-1.7B full fine-tune через FSDP FULL_SHARD на 2×T4 — entry point.

Запуск:
    torchrun --nproc_per_node=2 --master_port=29500 train_sft.py

CONFIG и training loop — здесь; всё остальное (model/data/checkpoint/utils)
импортируется из соседних модулей в той же папке. `torchrun` добавляет
директорию скрипта в `sys.path[0]`, поэтому `from data import ...` и т.п.
работают без дополнительной настройки PYTHONPATH.
"""
from __future__ import annotations

import os
from pathlib import Path

import torch
import torch.distributed as dist

from utils import Logger, set_seed
from data import build_dataloaders
from model import (
    build_optimizer_and_scheduler,
    evaluate,
    load_model_and_tokenizer,
    wrap_with_fsdp,
)
from checkpoint import load_checkpoint, save_checkpoint, save_final_hf


# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
SEED = 42
MAX_SEQ_LEN = 1024

def _find_jsonl(name):
    hits = list(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found in /kaggle/input")
    return hits[0]

TRAIN_JSONL = _find_jsonl("train.jsonl")
VAL_JSONL = _find_jsonl("val.jsonl")
OUT_DIR = Path("/kaggle/working")
CKPT_DIR = OUT_DIR / "checkpoints"
RESUME_DIR = Path("/kaggle/input/qwen-fft-sft-ckpt")

NUM_EPOCHS = 1
PER_DEVICE_BS = 1
GRAD_ACCUM = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

LOG_EVERY = 5
CHECKPOINT_EVERY_STEPS = 500
EVAL_EVERY_STEPS = 1000
EVAL_MAX_BATCHES = 50

CHATML_TEMPLATE = (
    "{% for m in messages %}"
    "<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)


# ═══════════════════════════════════════════════════════════════════════════
# TRAIN LOOP
# ═══════════════════════════════════════════════════════════════════════════

def train(model, optimizer, scheduler, train_loader, val_loader, train_sampler,
          rank, start_epoch, global_step, num_epochs, grad_accum, max_grad_norm,
          log_every, eval_every, checkpoint_every, eval_max_batches, ckpt_dir, logger):
    """Training loop с gradient accumulation, autocast(fp16) и FSDP-aware clip.
    Логи/eval/checkpoint триггерятся после каждого optimizer.step по интервалам.
    """
    model.train()
    steps_per_epoch = len(train_loader) // grad_accum

    for epoch in range(start_epoch, num_epochs):
        train_sampler.set_epoch(epoch)

        if epoch == start_epoch and global_step > 0:
            done_optim = global_step - start_epoch * steps_per_epoch
            skip_iter_steps = done_optim * grad_accum
        else:
            skip_iter_steps = 0

        for i, batch in enumerate(train_loader):
            if i < skip_iter_steps:
                continue
            if batch is None:
                continue

            batch = {k: v.to(rank, non_blocking=True) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                output = model(**batch)
                loss = output.loss / grad_accum
            loss.backward()

            if (i + 1) % grad_accum == 0:
                gnorm = model.clip_grad_norm_(max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step % log_every == 0:
                    logger.log_train(
                        global_step,
                        loss=loss.item() * grad_accum,
                        lr=scheduler.get_last_lr()[0],
                        gnorm=gnorm,
                    )

                if global_step % eval_every == 0:
                    val_loss = evaluate(model, val_loader, rank, eval_max_batches)
                    logger.log_eval(global_step, val_loss=val_loss)

                if global_step % checkpoint_every == 0:
                    save_checkpoint(
                        model, optimizer, scheduler,
                        global_step, epoch, rank, ckpt_dir,
                    )

    return logger.entries


# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main() -> None:
    dist.init_process_group("nccl")

    rank = int(os.environ["LOCAL_RANK"])
    world_size = dist.get_world_size()
    torch.cuda.set_device(rank)
    set_seed(SEED + rank)

    tokenizer, model = load_model_and_tokenizer(MODEL_NAME, CHATML_TEMPLATE)
    model = wrap_with_fsdp(model, rank)

    train_loader, val_loader, train_sampler = build_dataloaders(
        TRAIN_JSONL, VAL_JSONL, tokenizer, MAX_SEQ_LEN,
        PER_DEVICE_BS, SEED, rank, world_size,
    )

    steps_per_epoch = len(train_loader) // GRAD_ACCUM
    total_steps = steps_per_epoch * NUM_EPOCHS

    optimizer, scheduler = build_optimizer_and_scheduler(
        model, LR, WEIGHT_DECAY, WARMUP_RATIO, total_steps,
    )

    logger = Logger(rank, total_steps=total_steps, out_path=OUT_DIR / "metrics_train.json")

    global_step, start_epoch = 0, 0
    # Resume: ищем последний ckpt-step-*.pt в /kaggle/input/** (rglob — не зависит от
    # mount layout output'а: /kaggle/input/<slug>/checkpoints/ vs /kaggle/input/notebooks/<owner>/<slug>/checkpoints/).
    resume_ckpts = sorted(
        Path("/kaggle/input").rglob("ckpt-step-*.pt"),
        key=lambda p: int(p.stem.rsplit("-", 1)[-1]),
    )
    if resume_ckpts:
        if rank == 0:
            print(f"[resume] from {resume_ckpts[-1]}")
        global_step, start_epoch = load_checkpoint(
            model, optimizer, scheduler, resume_ckpts[-1], rank,
        )

    train(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=val_loader,
        train_sampler=train_sampler,
        rank=rank,
        start_epoch=start_epoch,
        global_step=global_step,
        num_epochs=NUM_EPOCHS,
        grad_accum=GRAD_ACCUM,
        max_grad_norm=MAX_GRAD_NORM,
        log_every=LOG_EVERY,
        eval_every=EVAL_EVERY_STEPS,
        checkpoint_every=CHECKPOINT_EVERY_STEPS,
        eval_max_batches=EVAL_MAX_BATCHES,
        ckpt_dir=CKPT_DIR,
        logger=logger,
    )

    save_checkpoint(
        model, optimizer, scheduler,
        global_step=total_steps, epoch=NUM_EPOCHS - 1,
        rank=rank, ckpt_dir=CKPT_DIR,
    )
    save_final_hf(model, tokenizer, OUT_DIR / "SFT-final-model", rank, ckpt_dir=CKPT_DIR)
    logger.save()
    logger.close()

    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main()


## 3. Запуск

Первый прогон — **smoke-test** на уменьшенном объёме: перед полным 50K временно подменить `TRAIN_JSONL` на `val.jsonl` в CONFIG (либо уменьшить dataset через `val.jsonl` как test-путь) и убедиться, что loss падает, peak_mem < 13.5 GB.

In [ ]:
!torchrun --nproc_per_node=2 --master_port=29500 /kaggle/working/src/train_sft.py

## 4. Инспекция артефактов

После `torchrun` на rank 0 будут:
- `/kaggle/working/checkpoints/ckpt-step-*.pt` — последние 2 чекпойнта (rotation keep_last=2 в `save_checkpoint`).
- `/kaggle/working/SFT-final-model/` — HF-формат для eval-ноутбука (~3.4 GB, fp16).
- `/kaggle/working/metrics_train.json` — список записей из Logger.

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("/kaggle/working/metrics_train.json").read_text())
print(f"entries: {len(metrics)}")
for m in metrics[-10:]:
    print(m)

print("\nArtifacts:")
!ls -la /kaggle/working/SFT-final-model/ 2>/dev/null
!ls -la /kaggle/working/checkpoints/ 2>/dev/null

## 5. Следующий шаг

`File → Save Version → Dataset`:
- **`kotshubin/qwen-fft-sft-final`** — только `SFT-final-model/` + `metrics_train.json` (~3.4 GB, для `3-qwen-fft-eval.ipynb`). Чекпойнты исключи вручную чтобы уложиться в 20 GB Dataset-лимит.
- **`kotshubin/qwen-fft-sft-ckpt`** (опционально, если нужен resume в следующей сессии) — последний `ckpt-step-*.pt` (~11 GB).